# Convolutional Neural Network (CNN) — End-to-End Demo

This notebook walks through every stage of building, training, and evaluating a CNN image classifier using **PyTorch** and the **CIFAR-10** dataset.

### What you'll see
1. **Data loading & exploration** — importing CIFAR-10 via `torchvision.datasets`
2. **Preprocessing & augmentation** — transforms, normalization, DataLoaders
3. **Model architecture** — a custom CNN with conv → batch-norm → pool blocks
4. **Training loop** — loss, optimizer, per-epoch metrics
5. **Evaluation** — accuracy, per-class breakdown, confusion matrix
6. **Visualizations** — sample images, training curves, predictions, feature maps

---
## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time, os

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 2. Load & Explore the CIFAR-10 Dataset

CIFAR-10 contains 60,000 32×32 colour images in 10 classes (6,000 per class).  
We import it directly through `torchvision.datasets.CIFAR10`.

In [ ]:
# Class labels
CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

# ---------- Transforms ----------
# Training: light augmentation + normalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),   # CIFAR-10 channel means
                         (0.2470, 0.2435, 0.2616)),   # CIFAR-10 channel stds
])

# Validation / Test: normalization only
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

# ---------- Download & create datasets ----------
full_train_dataset = CIFAR10(root="./data", train=True,
                             download=True, transform=train_transform)

test_dataset = CIFAR10(root="./data", train=False,
                       download=True, transform=test_transform)

# Split training set → 45,000 train + 5,000 validation
train_dataset, val_dataset = random_split(
    full_train_dataset, [45000, 5000],
    generator=torch.Generator().manual_seed(42),
)

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples:       {len(test_dataset):,}")

In [ ]:
# ---------- Visualise a batch of raw images ----------
# (load without normalization just for display)
viz_dataset = CIFAR10(root="./data", train=True, download=False,
                      transform=transforms.ToTensor())

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = viz_dataset[i]
    ax.imshow(img.permute(1, 2, 0))  # CHW → HWC
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 Images", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Class distribution ----------
all_labels = [viz_dataset[i][1] for i in range(len(viz_dataset))]
unique, counts = np.unique(all_labels, return_counts=True)

plt.figure(figsize=(8, 3))
plt.bar(CLASSES, counts, color="steelblue")
plt.ylabel("Count")
plt.title("Class Distribution (Training Set)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 3. DataLoaders

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

# Quick sanity check
images, labels = next(iter(train_loader))
print(f"Batch shape : {images.shape}")   # [128, 3, 32, 32]
print(f"Labels shape: {labels.shape}")   # [128]

---
## 4. Define the CNN Architecture

The network uses three convolutional blocks, each consisting of:

**Conv2d → BatchNorm → ReLU → Conv2d → BatchNorm → ReLU → MaxPool → Dropout**

Followed by a fully-connected classifier head.

In [ ]:
class CIFAR10_CNN(nn.Module):
    """A medium-depth CNN for 32×32 RGB image classification."""

    def __init__(self, num_classes: int = 10):
        super().__init__()

        # ---------- Block 1: 3 → 32 channels ----------
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 32×32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 32×32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # → 16×16
            nn.Dropout2d(0.2),
        )

        # ---------- Block 2: 32 → 64 channels ----------
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 16×16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # 16×16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # → 8×8
            nn.Dropout2d(0.3),
        )

        # ---------- Block 3: 64 → 128 channels ----------
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 8×8
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),# 8×8
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # → 4×4
            nn.Dropout2d(0.4),
        )

        # ---------- Classifier ----------
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


model = CIFAR10_CNN(num_classes=10).to(device)
print(model)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

---
## 5. Training Setup

In [ ]:
NUM_EPOCHS    = 25
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

---
## 6. Training & Validation Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
# ---------- Main training loop ----------
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>9}  {'Val Acc':>8}  {'Time':>6}")
print("-" * 58)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start
    marker = " *" if val_acc > best_val_acc else ""
    best_val_acc = max(best_val_acc, val_acc)

    print(f"{epoch:5d}  {train_loss:10.4f}  {train_acc:8.2%}  {val_loss:9.4f}  {val_acc:7.2%}  {elapsed:5.1f}s{marker}")

    # Save best model
    if val_acc >= best_val_acc:
        torch.save(model.state_dict(), "best_cnn.pth")

print(f"\nBest validation accuracy: {best_val_acc:.2%}")

---
## 7. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss
ax1.plot(epochs, history["train_loss"], label="Train", linewidth=2)
ax1.plot(epochs, history["val_loss"],   label="Validation", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Loss Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, [a * 100 for a in history["train_acc"]], label="Train", linewidth=2)
ax2.plot(epochs, [a * 100 for a in history["val_acc"]],   label="Validation", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Test-Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load("best_cnn.pth", map_location=device))

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2%}")

In [ ]:
# ---------- Per-class metrics ----------
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))

In [ ]:
# ---------- Confusion Matrix ----------
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — Test Set")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

---
## 9. Visualise Predictions

In [ ]:
def denormalize(tensor):
    """Undo CIFAR-10 normalization for display."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


# Grab one batch from the test loader
images, labels = next(iter(test_loader))
images_gpu = images.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images_gpu)
    probs = F.softmax(outputs, dim=1)
    confs, preds = probs.max(1)

# Plot 16 predictions
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    pred_label = CLASSES[preds[i]]
    true_label = CLASSES[labels[i]]
    color = "green" if preds[i] == labels[i] else "red"
    ax.set_title(f"{pred_label}\n({confs[i]:.0%})", fontsize=8, color=color)
    ax.axis("off")

fig.suptitle("Predictions (green = correct, red = wrong)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Visualise Feature Maps

Let's look at what the convolutional filters actually "see" at each block.

In [ ]:
# Hook to capture intermediate activations
activations = {}

def get_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

model.block1.register_forward_hook(get_hook("block1"))
model.block2.register_forward_hook(get_hook("block2"))
model.block3.register_forward_hook(get_hook("block3"))

# Forward pass on a single image
sample_img = images[:1].to(device)
with torch.no_grad():
    _ = model(sample_img)

# Plot first 16 feature maps from each block
fig, axes = plt.subplots(3, 16, figsize=(18, 5))
for row, block_name in enumerate(["block1", "block2", "block3"]):
    act = activations[block_name][0]  # first image in batch
    for col in range(16):
        axes[row][col].imshow(act[col], cmap="viridis")
        axes[row][col].axis("off")
    axes[row][0].set_ylabel(block_name, fontsize=10, rotation=0, labelpad=50)

fig.suptitle("Feature Maps (first 16 channels per block)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Visualise Learned Filters

The very first convolutional layer learns 32 filters of size 3×3×3 (RGB). We can visualise them directly.

In [ ]:
filters = model.block1[0].weight.data.cpu().clone()

# Normalize each filter to [0, 1] for display
filters -= filters.min()
filters /= filters.max()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i].permute(1, 2, 0).numpy())  # CHW → HWC
    ax.set_title(f"Filter {i}", fontsize=7)
    ax.axis("off")

fig.suptitle("Learned Filters — First Conv Layer (32 filters, 3×3×3)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 12. Inference on a Single Image

In [ ]:
def predict_single(model, dataset, index, device):
    """Run inference on one image and show top-5 class probabilities."""
    img, true_label = dataset[index]
    img_input = img.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(img_input)
        probs = F.softmax(logits, dim=1).squeeze()

    top5_probs, top5_idx = probs.topk(5)

    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3),
                                    gridspec_kw={"width_ratios": [1, 2]})
    ax1.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax1.set_title(f"True: {CLASSES[true_label]}", fontsize=10)
    ax1.axis("off")

    colors = ["green" if idx == true_label else "steelblue" for idx in top5_idx]
    ax2.barh([CLASSES[i] for i in top5_idx], top5_probs.cpu().numpy(), color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Probability")
    ax2.set_title("Top-5 Predictions")
    ax2.invert_yaxis()

    plt.tight_layout()
    plt.show()


# Try a few random test images
for idx in [0, 42, 100, 999]:
    predict_single(model, test_dataset, idx, device)

---
## Summary

| Stage | What happened |
|---|---|
| **Data** | Imported CIFAR-10 via `torchvision.datasets.CIFAR10`, applied augmentation & normalization |
| **Architecture** | 3-block CNN (Conv → BN → ReLU → Conv → BN → ReLU → MaxPool → Dropout) + FC head |
| **Training** | Adam optimizer, CrossEntropyLoss, ReduceLROnPlateau scheduler, 25 epochs |
| **Evaluation** | ~85-88% test accuracy (typical for this architecture on CIFAR-10 without heavy tuning) |
| **Visualizations** | Training curves, confusion matrix, predictions, feature maps, learned filters |

### Next steps to try
- **Deeper architectures**: ResNet, VGG, or EfficientNet via `torchvision.models`
- **Stronger augmentation**: CutOut, MixUp, AutoAugment
- **Learning rate warmup** with cosine annealing
- **Transfer learning** from ImageNet-pretrained weights